## ETL (Extract, Transform, Load) Pipeline

## Objectives

* Download the three raw CSV datasets (features.csv, sales.csv, and stores.csv) from Kaggle 

* Explore the data structure, relationship and overall quality of all three files

* Clean the data and combine the data into a single unified data structure

* Export the finalized, processed dataset into a new CSV file for analytic purposes

## Inputs

features data set.csv

sales data set.csv

stores data set.csv

The original files are stored in 'data/raw' and are not modified directly.

## Outputs

* Main output is to clean and transform the dataset and save it in 'data/cleaned'

The cleaned data will be used as input for the exploratory analysis and visualisation notebook.




---

# Section 1 - Extract

* The environment is set up by changing the working directory to the project root so a reliable relative file path is shown.

In [1]:
import os
import pandas as pd
import numpy as np

Locate the project root so relative data paths work whether the notebook starts in the project root, the notebook folder, or its parent directory.

In [2]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\munta\\Retail-Sales-Advanced-Analytics\\jupyter_notebooks'

In [3]:
from pathlib import Path

cwd = Path.cwd()
candidates = [cwd, *cwd.parents]

# If the kernel starts one folder above the project, check its direct subfolders too.
if not any((candidate / "data" / "raw").is_dir() for candidate in candidates):
    candidates.extend(candidate for candidate in cwd.iterdir() if candidate.is_dir())

project_root = next(
    (candidate for candidate in candidates if (candidate / "data" / "raw").is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not locate the project root containing data/raw.")

os.chdir(project_root)
print(f"Working directory set to: {project_root}")

Working directory set to: c:\Users\munta\Retail-Sales-Advanced-Analytics


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\munta\\Retail-Sales-Advanced-Analytics'

Check raw dataset files in the target dataset/raw folder.

In [5]:
import os

files = os.listdir("data/raw")
print(files)

['Features data set.csv.csv', 'sales_data_set.csv', 'stores data-set.csv.csv']


The three CSV files are loaded into individual Pandas Dataframes

In [6]:
# Read raw dataset into dataframes
stores = pd.read_csv("data/raw/stores data-set.csv.csv")
features = pd.read_csv("data/raw/Features data set.csv.csv")
sales = pd.read_csv("data/raw/sales_data_set.csv")

---

In [7]:
# Performing data validation
for name, df in [("sales", sales), ("features", features),("stores", stores)]:
    print(f"\n{name} dataset")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Missing values: {df.isnull().sum().sum():,}")
    print(f"Duplicate rows: {df.duplicated().sum():,}")



sales dataset
Rows: 421,570
Columns: 5
Missing values: 0
Duplicate rows: 0

features dataset
Rows: 8,190
Columns: 12
Missing values: 24,040
Duplicate rows: 0

stores dataset
Rows: 45
Columns: 3
Missing values: 0
Duplicate rows: 0


In [8]:
# Identifying the columns with missing values in the features dataset
features.isnull().sum()

Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4158
MarkDown2       5269
MarkDown3       4577
MarkDown4       4726
MarkDown5       4140
CPI              585
Unemployment     585
IsHoliday          0
dtype: int64

In [9]:
# Checking the remaining columns
features["Unemployment"].isnull().sum()

585

In [10]:
# Check whether CPI and Unemployment are missing together
both_missing = (
    features["CPI"].isnull()
    & features["Unemployment"].isnull()
)

print("Both missing:", both_missing.sum())
print("Only CPI missing:", (features["CPI"].isnull() & ~features["Unemployment"].isnull()).sum())
print("Only Unemployment missing:", (~features["CPI"].isnull() & features["Unemployment"].isnull()).sum())

Both missing: 585
Only CPI missing: 0
Only Unemployment missing: 0


In [11]:
# Examine records where CPI and Unemployment are missing
missing_economic = features[
    features["CPI"].isnull()
]

print("Affected stores:", missing_economic["Store"].nunique())
print("Earliest date:", missing_economic["Date"].min())
print("Latest date:", missing_economic["Date"].max())

display(missing_economic.head())

Affected stores: 45
Earliest date: 03/05/2013
Latest date: 31/05/2013


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
169,1,03/05/2013,66.66,3.386,2298.63,2.00,129.90,55.46,1301.04,NaN,NaN,False
170,1,10/05/2013,63.90,3.392,4624.61,83.88,42.38,1618.31,8144.90,NaN,NaN,False
171,1,17/05/2013,69.53,3.454,16170.50,92.00,8.32,4127.24,6206.97,NaN,NaN,False
172,1,24/05/2013,77.19,3.494,7959.89,178.00,1621.47,3152.57,2938.70,NaN,NaN,False
173,1,31/05/2013,78.02,3.466,2369.77,142.45,475.35,45.55,2056.84,NaN,NaN,False


In [12]:
# Check the date range of the sales dataset
sales_dates = pd.to_datetime(sales["Date"], dayfirst=True)

print("First sales date:", sales_dates.min())
print("Last sales date:", sales_dates.max())

First sales date: 2010-02-05 00:00:00
Last sales date: 2012-10-26 00:00:00


In [13]:
# Create a copy of features for the sales analysis period
features_clean = features.copy()

features_clean["Date"] = pd.to_datetime(
    features_clean["Date"],
    dayfirst=True
)

# Keep dates within the sales period
features_clean = features_clean[
    features_clean["Date"].between(
        sales_dates.min(),
        sales_dates.max()
    )
].copy()

# Check missing values in the relevant records
print(features_clean.isnull().sum())

Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4155
MarkDown2       4798
MarkDown3       4389
MarkDown4       4470
MarkDown5       4140
CPI                0
Unemployment       0
IsHoliday          0
dtype: int64


In [14]:
# Check missing markdown values by year
markdown_cols = [
    "MarkDown1", "MarkDown2", "MarkDown3",
    "MarkDown4", "MarkDown5"
]

features_clean["Year"] = features_clean["Date"].dt.year

print(
    features_clean.groupby("Year")[markdown_cols]
    .apply(lambda df: df.isnull().sum())
)

      MarkDown1  MarkDown2  MarkDown3  MarkDown4  MarkDown5
Year                                                       
2010       2160       2160       2160       2160       2160
2011       1986       2047       1998       2038       1980
2012          9        591        231        272          0


In [15]:
# Check whether markdown information is available
features_clean["Markdown_Data_Available"] = (
    features_clean[markdown_cols].notna().any(axis=1)
)

features_clean["Markdown_Data_Available"].value_counts()

Markdown_Data_Available
False    4140
True     2295
Name: count, dtype: int64

A Total_MarkDown column was created by adding together the five markdown columns. Missing values were preserved where no markdown information was available.

In [16]:
# Calculate total recorded markdown amounts
features_clean["Total_MarkDown"] = (
    features_clean[markdown_cols].sum(axis=1, min_count=1)
)

# Check the results
features_clean[
    ["Date", "Markdown_Data_Available", "Total_MarkDown"]
].head()

,Date,Markdown_Data_Available,Total_MarkDown
0,2010-02-05,False,NaN
1,2010-02-12,False,NaN
2,2010-02-19,False,NaN
3,2010-02-26,False,NaN
4,2010-03-05,False,NaN


Missing markdown values were retained as NaN because missing information does not necessarily mean that no promotion occurred.

In [17]:
# Check totals for records with and without markdown data
print("Missing totals:", features_clean["Total_MarkDown"].isna().sum())
print("Recorded totals:", features_clean["Total_MarkDown"].notna().sum())

# Display examples where markdown information exists
display(
    features_clean.loc[
        features_clean["Markdown_Data_Available"],
        ["Date", "Total_MarkDown"]
    ].head()
)

Missing totals: 4140
Recorded totals: 2295


,Date,Total_MarkDown
92,2011-11-11,25671.68
93,2011-11-18,12796.45
94,2011-11-25,56876.74
95,2011-12-02,29655.58
96,2011-12-09,22865.91


In [18]:
# Create a copy of the original sales data
sales_clean = sales.copy()

# Convert Date into datetime format
sales_clean["Date"] = pd.to_datetime(
    sales_clean["Date"],
    dayfirst=True
)

# Check the result
print(sales_clean["Date"].dtype)
print("Sales rows:", len(sales_clean))

datetime64[ns]
Sales rows: 421570


In [19]:
# Create date columns for the cleaned dataset.
sales_clean["Year"] = sales_clean["Date"].dt.year
sales_clean["Month"] = sales_clean["Date"].dt.month
sales_clean["Week"] = sales_clean["Date"].dt.isocalendar().week.astype(int)

In [20]:
#Printing the shape of the dataframes
print("Stores:", stores.shape)
print("Features:", features.shape)
print("Sales:", sales.shape)

Stores: (45, 3)
Features: (8190, 12)
Sales: (421570, 5)


# Section 2 - Transform

This section will inspect and clean data by converting data types, extract temporal features, handle missing values, and merge all thre datasets into one unified structure

In [21]:
#
stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


---

In [22]:
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,05/02/2010,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,12/02/2010,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,19/02/2010,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,26/02/2010,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,05/03/2010,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [23]:
sales.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,05/02/2010,24924.50,False
1,1,1,12/02/2010,46039.49,True
2,1,1,19/02/2010,41595.55,False
3,1,1,26/02/2010,19403.54,False
4,1,1,05/03/2010,21827.90,False


In [24]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Store   45 non-null     int64 
 1   Type    45 non-null     object
 2   Size    45 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.2+ KB


In [25]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   object 
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921 non-null   float64
 6   MarkDown3     3613 non-null   float64
 7   MarkDown4     3464 non-null   float64
 8   MarkDown5     4050 non-null   float64
 9   CPI           7605 non-null   float64
 10  Unemployment  7605 non-null   float64
 11  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(1), object(1)
memory usage: 712.0+ KB


In [26]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB


* Column names

In [27]:
print("Stores columns:")
print(stores.columns.tolist())

print("\nFeatures columns:")
print(features.columns.tolist())

print("\nSales columns:")
print(sales.columns.tolist())

Stores columns:
['Store', 'Type', 'Size']

Features columns:
['Store', 'Date', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'IsHoliday']

Sales columns:
['Store', 'Dept', 'Date', 'Weekly_Sales', 'IsHoliday']


* Missing values

In [28]:
# Check for null values in the dataframes
print("Stores missing values")
display(stores.isnull().sum())

print("Features missing values")
display(features.isnull().sum())

print("Sales missing values")
display(sales.isnull().sum())

Stores missing values


Store    0
Type     0
Size     0
dtype: int64

Features missing values


Store              0
Date               0
Temperature        0
Fuel_Price         0
MarkDown1       4158
MarkDown2       5269
MarkDown3       4577
MarkDown4       4726
MarkDown5       4140
CPI              585
Unemployment     585
IsHoliday          0
dtype: int64

Sales missing values


Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64

In [29]:
# Check for duplicates in the individual dataframes
print("Stores duplicates:", stores.duplicated().sum())
print("Features duplicates:", features.duplicated().sum())
print("Sales duplicates:", sales.duplicated().sum())

Stores duplicates: 0
Features duplicates: 0


Sales duplicates: 0


In [30]:
# Check for duplicates in the sales dataframe to avoid a structural issue in the data
sales.duplicated(
    subset=["Store", "Dept", "Date"]
).sum()

0

* Dates

In [31]:
# check the data types of both dataframes
print(features["Date"].dtype)
print(sales["Date"].dtype)

object
object


In [32]:
# Convert the "Date" column in both features and sales dataframes to datetime format
features["Date"] = pd.to_datetime(features["Date"], format="%d/%m/%Y").astype("datetime64[ns]")
sales["Date"] = pd.to_datetime(sales["Date"], format="%d/%m/%Y").astype("datetime64[ns]")

#verify the data types after conversion
print(features["Date"].dtype)
print(sales["Date"].dtype)

datetime64[ns]
datetime64[ns]


* Numeric columns

In [33]:
#Inspect before conversion
print(features.dtypes)

Store                    int64
Date            datetime64[ns]
Temperature            float64
Fuel_Price             float64
MarkDown1              float64
MarkDown2              float64
MarkDown3              float64
MarkDown4              float64
MarkDown5              float64
CPI                    float64
Unemployment           float64
IsHoliday                 bool
dtype: object


In [34]:
print(sales.dtypes)

Store                    int64
Dept                     int64
Date            datetime64[ns]
Weekly_Sales           float64
IsHoliday                 bool
dtype: object


* Categorical values

In [35]:
# For store types
print(stores["Type"].value_counts())

Type
A    22
B    17
C     6
Name: count, dtype: int64


In [36]:
# For sales 
print(sales["Store"].nunique())
print(sales["Dept"].nunique())

45
81


In [37]:
# For holiday flags
print(features["IsHoliday"].value_counts(dropna=False))
print(sales["IsHoliday"].value_counts(dropna=False))

IsHoliday
False    7605
True      585
Name: count, dtype: int64
IsHoliday
False    391909
True      29661
Name: count, dtype: int64


These codes are there to standardize and check categorical data

In [38]:
features.describe().T

,count,mean,min,25%,50%,75%,max,std
Store,8190.0,23.0,1.0,12.0,23.0,34.0,45.0,12.987966
Date,8190,2011-10-31 12:00:00,2010-02-05 00:00:00,2010-12-17 00:00:00,2011-10-31 12:00:00,2012-09-14 00:00:00,2013-07-26 00:00:00,NaN
Temperature,8190.0,59.356198,-7.29,45.9025,60.71,73.88,101.95,18.678607
Fuel_Price,8190.0,3.405992,2.472,3.041,3.513,3.743,4.468,0.431337
MarkDown1,4032.0,7032.371786,-2781.45,1577.5325,4743.58,8923.31,103184.98,9262.747448
MarkDown2,2921.0,3384.176594,-265.76,68.88,364.57,2153.35,104519.54,8793.583016
MarkDown3,3613.0,1760.10018,-179.26,6.6,36.26,163.15,149483.31,11276.462208
MarkDown4,3464.0,3292.935886,0.22,304.6875,1176.425,3310.0075,67474.85,6792.329861
MarkDown5,4050.0,4132.216422,-185.17,1440.8275,2727.135,4832.555,771448.1,13086.690278
CPI,7605.0,172.460809,126.064,132.364839,182.764003,213.932412,228.976456,39.738346


In [39]:
sales.describe().T

,count,mean,min,25%,50%,75%,max,std
Store,421570.0,22.200546,1.0,11.0,22.0,33.0,45.0,12.785297
Dept,421570.0,44.260317,1.0,18.0,37.0,74.0,99.0,30.492054
Date,421570,2011-06-18 08:30:31.963375104,2010-02-05 00:00:00,2010-10-08 00:00:00,2011-06-17 00:00:00,2012-02-24 00:00:00,2012-10-26 00:00:00,NaN
Weekly_Sales,421570.0,15981.258123,-4988.94,2079.65,7612.03,20205.8525,693099.36,22711.183519


* Outliers

In [40]:
print("Negative weekly sales:", (sales["Weekly_Sales"] < 0).sum())
print(sales["Weekly_Sales"].describe())

Negative weekly sales: 1285
count    421570.000000
mean      15981.258123
std       22711.183519
min       -4988.940000
25%        2079.650000
50%        7612.030000
75%       20205.852500
max      693099.360000
Name: Weekly_Sales, dtype: float64


Extreme sales values were looked into instead of being eliminated.              

High weekly sales might indicate store or department performance making them important to business requirements. 

Records were only taken into consideration for deletion if they were identified as invalid or blatantly incorrect.

Transform

In [41]:
sales_clean[["Date", "Year", "Month", "Week"]].head()

,Date,Year,Month,Week
0,2010-02-05,2010,2,5
1,2010-02-12,2010,2,6
2,2010-02-19,2010,2,7
3,2010-02-26,2010,2,8
4,2010-03-05,2010,3,9


In [42]:
markdown_cols = [
    "MarkDown1",
    "MarkDown2",
    "MarkDown3",
    "MarkDown4",
    "MarkDown5"
]

features[markdown_cols].describe()

,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5
count,4032.000000,2921.000000,3613.000000,3464.000000,4050.000000
mean,7032.371786,3384.176594,1760.100180,3292.935886,4132.216422
std,9262.747448,8793.583016,11276.462208,6792.329861,13086.690278
min,-2781.450000,-265.760000,-179.260000,0.220000,-185.170000
25%,1577.532500,68.880000,6.600000,304.687500,1440.827500
50%,4743.580000,364.570000,36.260000,1176.425000,2727.135000
75%,8923.310000,2153.350000,163.150000,3310.007500,4832.555000
max,103184.980000,104519.540000,149483.310000,67474.850000,771448.100000


In [43]:
features[markdown_cols].isnull().sum()

MarkDown1    4158
MarkDown2    5269
MarkDown3    4577
MarkDown4    4726
MarkDown5    4140
dtype: int64

In [44]:
features[markdown_cols].notna().sum()

MarkDown1    4032
MarkDown2    2921
MarkDown3    3613
MarkDown4    3464
MarkDown5    4050
dtype: int64

---


#  Merge the three datasets into one!
The cleaned sales, features and stores datasets will be merged into one clean dataset. The sales datasets specifies the weekly department sales records, whilst features provides regional and promotional information, and lastly stores demonstrates store characteristics.

In [45]:
# Merge cleaned sales with features
df = sales_clean.merge(
    features_clean.drop(columns=["IsHoliday", "Year"]),
    on=["Store", "Date"],
    how="left",
    validate="many_to_one"
)

print("Original sales rows:", len(sales_clean))
print("Merged rows:", len(df))


Original sales rows: 421570
Merged rows: 421570


In [46]:
# Merge store information into the combined dataset
df = df.merge(
    stores,
    on="Store",
    how="left",
    validate="many_to_one"
)

print("Final merged rows:", len(df))
print("Missing store types:", df["Type"].isnull().sum())
print("Missing store sizes:", df["Size"].isnull().sum())

Final merged rows: 421570
Missing store types: 0
Missing store sizes: 0


In [47]:
# Classify markdown activity without treating missing data as zero
df["Markdown_Activity"] = pd.NA

df.loc[df["Total_MarkDown"] == 0, "Markdown_Activity"] = "No recorded markdown"
df.loc[df["Total_MarkDown"] > 0, "Markdown_Activity"] = "Recorded markdown"

print(df["Markdown_Activity"].value_counts(dropna=False))

Markdown_Activity
<NA>                 270138
Recorded markdown    151432
Name: count, dtype: int64


A Markdown_Activity column was created to show where markdowns were recorded. I kept missing values separate because missing data doesn't necessarily mean there were no promotions.

In [48]:
# Check the shape of the merged dataset
print("Merged dataset shape:", df.shape)


Merged dataset shape: (421570, 22)


In [49]:
# Check whether the merged data loaded correctly
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Year,Month,Week,Temperature,Fuel_Price,...,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Markdown_Data_Available,Total_MarkDown,Type,Size,Markdown_Activity
0,1,1,2010-02-05,24924.50,False,2010,2,5,42.31,2.572,...,NaN,NaN,NaN,211.096358,8.106,False,NaN,A,151315,<NA>
1,1,1,2010-02-12,46039.49,True,2010,2,6,38.51,2.548,...,NaN,NaN,NaN,211.242170,8.106,False,NaN,A,151315,<NA>
2,1,1,2010-02-19,41595.55,False,2010,2,7,39.93,2.514,...,NaN,NaN,NaN,211.289143,8.106,False,NaN,A,151315,<NA>
3,1,1,2010-02-26,19403.54,False,2010,2,8,46.63,2.561,...,NaN,NaN,NaN,211.319643,8.106,False,NaN,A,151315,<NA>
4,1,1,2010-03-05,21827.90,False,2010,3,9,46.50,2.625,...,NaN,NaN,NaN,211.350143,8.106,False,NaN,A,151315,<NA>


In [50]:
# Validate the final merged dataset
print("Dataset shape:", df.shape)

print("\nMissing values by column:")
print(df.isnull().sum())

print("\nDuplicate store-department-date records:")
print(df.duplicated(subset=["Store", "Dept", "Date"]).sum())

Dataset shape: (421570, 22)

Missing values by column:
Store                           0
Dept                            0
Date                            0
Weekly_Sales                    0
IsHoliday                       0
Year                            0
Month                           0
Week                            0
Temperature                     0
Fuel_Price                      0
MarkDown1                  270889
MarkDown2                  310322
MarkDown3                  284479
MarkDown4                  286603
MarkDown5                  270138
CPI                             0
Unemployment                    0
Markdown_Data_Available         0
Total_MarkDown             270138
Type                            0
Size                            0
Markdown_Activity          270138
dtype: int64

Duplicate store-department-date records:


0


In [51]:
# Check for duplicates in the merged dataframe
df.duplicated(
    subset=["Store", "Dept", "Date"]
).sum()

0

# Load - Export

* The cleaned dataset is exported to the data/cleaned folder as a CSV file. This file will be used for analysis, machine learning and the Power BI dashboard.

In [52]:
os.makedirs("data/cleaned", exist_ok=True)


In [53]:
# Export the cleaned dataset to the data/cleaned folder.
df.to_csv("data/cleaned/retail_sales_cleaned.csv", index=False)

print("Cleaned dataset exported successfully.")

Cleaned dataset exported successfully.


In [54]:
# Last quality data quality check.
print("Final rows:", len(df))
print("Final columns:", df.shape[1])
print("Duplicate sales records:", df.duplicated(
    subset=["Store", "Dept", "Date"]
).sum())
print("Date range:", df["Date"].min(), "to", df["Date"].max())

Final rows: 421570
Final columns: 22
Duplicate sales records: 0
Date range: 2010-02-05 00:00:00 to 2012-10-26 00:00:00
